# Synthetic Figure Reproduction

This notebook drives the refactored C/C++ simulator and reproduces synthetic-data figures from Angiolelli et al., with emphasis on Fig.5 avalanche size and duration distributions.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "python"))

import matplotlib.pyplot as plt
import pandas as pd

from criticality_analysis import avalanche_table, find_output_file, fit_power_law, load_medie, load_q, load_rate, load_spikes, run_simulation
from criticality_analysis.plotting import plot_avalanche_distribution

FIG_DIR = PROJECT / "results" / "figures"
TABLE_DIR = PROJECT / "results" / "tables"
RUN_ROOT = PROJECT / "results" / "runs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

EXECUTABLE = PROJECT / "build" / ("criticality_sim.exe" if sys.platform.startswith("win") else "criticality_sim")
EXECUTABLE

## Run Simulations

Set `RUN_SIMULATIONS = True` after building the C/C++ target. The Fig.5 configs use the paper scale (`N=13200`, `P=20`) and `bin=5` for 5 ms module-rate windows.

In [ ]:
RUN_SIMULATIONS = False

stationary_bin5_experiments = {
    "E0=5.7, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e57_i11_bin5.seed",
    "E0=6.8, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e68_i11_bin5.seed",
    "E0=6.9, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e69_i11_bin5.seed",
}

if RUN_SIMULATIONS:
    for label, seed in stationary_bin5_experiments.items():
        run_name = seed.stem
        print(f"Running {label}: {run_name}")
        run_simulation(EXECUTABLE, seed, RUN_ROOT, run_name=run_name)
else:
    print("Skipping simulator runs. Existing outputs under results/runs will be used.")

## Fig.5: Synthetic Neuronal Avalanches

Definition from the paper: an avalanche is a continuous interval in which at least one module's rate is greater than zero. The module rate is computed over 5 ms bins. Size is the integral of summed module rates; duration is the active interval length.

In [ ]:
avalanches = {}
for label, seed in stationary_bin5_experiments.items():
    run_dir = RUN_ROOT / seed.stem
    rate_path = find_output_file(run_dir, "rate3", seed.stem)
    rate = load_rate(rate_path)
    avalanches[label] = avalanche_table(rate)
    print(label, len(avalanches[label]), "avalanches", "from", rate_path)

fit_label = "E0=6.9, I0=1.1"
size_fit = fit_power_law(avalanches[fit_label]["size"])
duration_fit = fit_power_law(avalanches[fit_label]["duration_ms"])

fit_table = pd.DataFrame([
    {"quantity": "size", **size_fit.__dict__},
    {"quantity": "duration_ms", **duration_fit.__dict__},
])
fit_table.to_csv(TABLE_DIR / "fig5_powerlaw_fits.csv", index=False)
fit_table

In [ ]:
plot_avalanche_distribution(
    avalanches,
    "size",
    fit_label=fit_label,
    fit=size_fit,
    output=FIG_DIR / "fig5a_avalanche_size.png",
)
plt.show()

plot_avalanche_distribution(
    avalanches,
    "duration_ms",
    fit_label=fit_label,
    fit=duration_fit,
    output=FIG_DIR / "fig5b_avalanche_duration.png",
)
plt.show()

## Synthetic Fig.2-Fig.4 Helpers

The cells below read the same simulator outputs used by the original program. They provide the building blocks for rate, Fano/CV, overlap, raster, and module-rate panels. Run parameter sweeps with additional `SEED` files, then point `run_name` at each output directory.

In [ ]:
def load_run_outputs(run_name, name=None, pout=3):
    name = name or run_name
    run_dir = RUN_ROOT / run_name
    outputs = {}
    for key, prefix, loader in [
        ("medie", "medie3", load_medie),
        ("q", "q3", lambda p: load_q(p, pout=pout)),
        ("rate", "rate3", load_rate),
        ("spikes", "spikes3", lambda p: load_spikes(p, pout=pout)),
    ]:
        try:
            outputs[key] = loader(find_output_file(run_dir, prefix, name))
        except FileNotFoundError as exc:
            print(exc)
    return outputs

example = load_run_outputs("stationary_e69_i11_bin5")
example.keys()

In [ ]:
if "medie" in example:
    ax = example["medie"].plot(x="time_ms", y=["rate_hz", "fano", "cv"], subplots=True, figsize=(7, 5), legend=True)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "synthetic_rate_fano_cv_timeseries.png", dpi=180)
    plt.show()

if "q" in example:
    q_cols = [c for c in example["q"].columns if c.startswith("q_")]
    example["q"].plot(x="t_end", y=["q_max"] + q_cols[:3], figsize=(7, 3))
    plt.axhline(0.8, color="black", linestyle="--", linewidth=1)
    plt.ylabel("overlap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "synthetic_overlap_timeseries.png", dpi=180)
    plt.show()

if "rate" in example:
    module_cols = [c for c in example["rate"].columns if c.startswith("module_")]
    plt.figure(figsize=(7, 4))
    plt.imshow(example["rate"][module_cols].T, aspect="auto", origin="lower", interpolation="nearest")
    plt.xlabel("time bin")
    plt.ylabel("module")
    plt.colorbar(label="rate")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "synthetic_module_rate_heatmap.png", dpi=180)
    plt.show()